# HW-03: Classification — ML Zoomcamp 2026

Goal: predict `converted`, indicating whether a prospective customer signs up for the platform.

[Homework instructions](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md) · [Module materials](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/cohorts/2026/03-classification)

This notebook is self-contained and works locally or in Google Colab. All intermediate tables are recalculated.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivazin/ml-zoomcamp/blob/main/hw-03/homework_03.ipynb)

## Runtime setup

Use **Runtime → Run all** in Colab. Missing Python packages are installed automatically, and the dataset is downloaded from the course repository. No other files from this project are required. An internet connection is needed for the first run.

In [1]:
# Install missing dependencies in a fresh notebook runtime.
import importlib.util
import subprocess
import sys

requirements = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn'}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [2]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
DATA_PATH = Path("data/course_lead_scoring_2026.csv")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    with urlopen(DATA_URL, timeout=60) as response:
        DATA_PATH.write_bytes(response.read())

# Check that the downloaded file matches the release used for these answers.
EXPECTED_SHA256 = "2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9"
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256
print("Dataset ready:", DATA_PATH)


Dataset ready: data/course_lead_scoring_2026.csv


In [3]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

def prepare():
    # keep_default_na=False preserves the literal NA category;
    # empty fields are still treated as missing values.
    raw = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
    categorical = raw.select_dtypes(include=["object", "str"]).columns.tolist()
    numerical = [c for c in raw.columns if c not in categorical and c != "converted"]
    df = raw.copy()
    df[categorical] = df[categorical].fillna("NA")
    df[numerical] = df[numerical].fillna(0.0)
    full_train, test = train_test_split(df, test_size=0.2, random_state=42)
    train, val = train_test_split(full_train, test_size=0.25, random_state=42)
    y_train, y_val = train.converted.to_numpy(), val.converted.to_numpy()
    train, val, test = [part.drop(columns="converted") for part in (train, val, test)]
    return raw, df, categorical, numerical, train, val, test, y_train, y_val


def fit_accuracy(train, val, y_train, y_val, features, C=1.0):
    # Fit the encoder only on the training set. Keep numerical features unscaled:
    # this preserves the homework setup and the effect of C on the original scale.
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train[features].to_dict(orient="records"))
    X_val = dv.transform(val[features].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict_proba(X_val)[:, 1] >= 0.5
    return float(accuracy_score(y_val, predictions))


raw, df, categorical, numerical, train, val, test, y_train, y_val = prepare()
print("Shape:", df.shape)
print("Categorical features:", categorical)
print("Numerical features:", numerical)
df.head()

Shape: (5000, 9)
Categorical features: ['lead_source', 'industry', 'employment_status', 'location']
Numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Replace missing categories with the string `NA` and missing numbers with `0.0`. These are constants, so applying them before splitting does not use validation or test statistics. Preserve the literal string `NA` when reading the CSV. Check missing values, the binary target, and split sizes.

In [4]:
display(raw.isna().sum().rename("missing_before").to_frame())
assert not df.isna().any().any()
assert set(df.converted.unique()) == {0, 1}
print("train / validation / test:", len(train), len(val), len(test))
assert set(train.index).isdisjoint(val.index)
assert set(train.index).isdisjoint(test.index)
assert set(val.index).isdisjoint(test.index)
assert "converted" not in train.columns

,missing_before
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


train / validation / test: 3000 1000 1000


## Q1. Mode of industry

The mode is the most frequent value. Count observations in the full prepared dataset.

In [5]:
display(df.industry.value_counts().rename("count").to_frame())
print("Answer Q1:", df.industry.mode().iloc[0])

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463
NA,240


Answer Q1: technology


## Q2. Correlations between numerical features

Pearson correlation measures linear association: values near +1 indicate a strong positive relationship, values near −1 a strong negative relationship, and values near 0 a weak linear relationship. Exclude the target and diagonal entries when comparing feature pairs. Select the answer only from the four pairs in the instructions. Correlation alone does not establish causation.

In [6]:
corr = df[numerical].corr()
display(corr)
pairs = [("interaction_count", "lead_score"),
         ("number_of_courses_viewed", "lead_score"),
         ("number_of_courses_viewed", "interaction_count"),
         ("annual_income", "interaction_count")]
pair_scores = pd.Series({f"{a} / {b}": corr.loc[a, b] for a, b in pairs})
display(pair_scores.sort_values(ascending=False).rename("correlation").to_frame())
print("Answer Q2:", pair_scores.idxmax())

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


,correlation
interaction_count / lead_score,0.915746
number_of_courses_viewed / lead_score,0.757204
number_of_courses_viewed / interaction_count,0.721609
annual_income / interaction_count,0.122842


Answer Q2: interaction_count / lead_score


## Data splitting and Q3. Mutual information

Use two `train_test_split` calls with `random_state=42`: first reserve 20% for testing, then reserve 25% of the remaining data for validation. This gives a 60% / 20% / 20% split. Remove the target from the features.

Mutual information measures dependence between a categorical feature and the target class. A larger value means the feature is more informative on its own; it does not measure the full model's performance. Use only the training set and round scores to two decimal places.

In [7]:
mi = pd.Series({c: mutual_info_score(train[c], y_train) for c in categorical})
mi_rounded = mi.map(lambda score: round(score, 2))
display(pd.DataFrame({"MI": mi, "rounded": mi_rounded}).sort_values("MI", ascending=False))
print("Answer Q3:", mi_rounded.idxmax())

,MI,rounded
lead_source,0.034346,0.03
employment_status,0.018966,0.02
industry,0.001995,0.00
location,0.001043,0.00


Answer Q3: lead_source


## Q4. Logistic regression

`DictVectorizer` creates an indicator for each categorical value. For example, `industry=technology` receives 1 in the corresponding category column and 0 in the others. Numerical values retain their original scale. Fit the encoder on the training set, then apply it to validation.

Logistic regression converts a linear combination of features into a probability using the sigmoid function. Predict class 1 when the probability is ≥ 0.5. Accuracy is the proportion of predictions that match the actual class.

`fit_accuracy` uses `solver='liblinear', C=1.0, max_iter=1000, random_state=42`. Include all eight features without scaling or additional transformations. A dense matrix (`sparse=False`) is practical for this dataset and compatible with liblinear.

In [8]:
features = train.columns.tolist()
baseline = fit_accuracy(train, val, y_train, y_val, features)
print("Unrounded accuracy:", baseline)
print("Answer Q4:", round(baseline, 2))

Unrounded accuracy: 0.645
Answer Q4: 0.65


## Q5. Feature elimination

Retrain the model after removing each original feature in turn, including all of its one-hot columns. Calculate `difference = baseline_accuracy - accuracy_without_feature` without rounding.

A positive difference means removing the feature reduces accuracy; a negative difference means it improves accuracy. Select the smallest signed difference among the answer options, rather than the smallest absolute difference. This experiment describes this particular model and split, not a feature's universal value.

In [9]:
rows = []
for feature in features:
    accuracy = fit_accuracy(train, val, y_train, y_val, [c for c in features if c != feature])
    rows.append({"feature": feature, "accuracy_without": accuracy, "difference": baseline - accuracy})
elimination = pd.DataFrame(rows).set_index("feature")
display(elimination)
candidates = ["lead_source", "number_of_courses_viewed", "interaction_count"]
print("Answer Q5:", elimination.loc[candidates, "difference"].idxmin())

,accuracy_without,difference
feature,,
lead_source,0.642,0.003
industry,0.645,0.000
employment_status,0.645,0.000
location,0.645,0.000
annual_income,0.724,-0.079
number_of_courses_viewed,0.643,0.002
interaction_count,0.601,0.044
lead_score,0.644,0.001


Answer Q5: number_of_courses_viewed


## Q6. Regularization

`C` controls the inverse strength of regularization: a smaller `C` means a stronger penalty on large coefficients. Evaluate all four values using the same training and validation sets. Compare accuracy rounded to three decimal places; break ties by selecting the smaller `C`.

Keep features on their original scale to follow the homework setup. In other tasks, scaling can substantially change regularization results. Do not use the test set to select parameters.

In [10]:
rows = []
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    accuracy = fit_accuracy(train, val, y_train, y_val, features, C=C)
    rows.append({"C": C, "accuracy": accuracy, "rounded": round(accuracy, 3)})
regularization = pd.DataFrame(rows)
display(regularization)
best_C = regularization.sort_values(["rounded", "C"], ascending=[False, True]).iloc[0]["C"]
print("Answer Q6:", best_C)

,C,accuracy,rounded
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


Answer Q6: 0.001


## Answers

| Question | Answer |
|---|---|
| Q1 | technology |
| Q2 | interaction_count and lead_score |
| Q3 | lead_source |
| Q4 | 0.65 |
| Q5 | number_of_courses_viewed |
| Q6 | 0.001 |

The final cell verifies the answers and exports computed values to `notebook_results.json`. The full correlation matrix is saved in `correlation_matrix.csv`. Running `solution.py` separately produces `results.json`, including library versions and the dataset SHA-256.

In [11]:
# Export results from this notebook without requiring a local reference file.
import json

answers = {
    "Q1": df.industry.mode().iloc[0],
    "Q2": pair_scores.idxmax(),
    "Q3": mi_rounded.idxmax(),
    "Q4": round(baseline, 2),
    "Q5": elimination.loc[candidates, "difference"].idxmin(),
    "Q6": float(best_C),
}
expected_answers = {
    "Q1": "technology", "Q2": "interaction_count / lead_score",
    "Q3": "lead_source", "Q4": 0.65,
    "Q5": "number_of_courses_viewed", "Q6": 0.001,
}
assert answers == expected_answers
results = {
    "answers": answers,
    "baseline_accuracy": baseline,
    "correlations": pair_scores.to_dict(),
    "mutual_information": mi.to_dict(),
    "elimination": elimination.to_dict(orient="index"),
    "regularization": regularization.to_dict(orient="records"),
}
# Keep notebook exports separate from the script's results.json.
Path("notebook_results.json").write_text(json.dumps(results, indent=2) + "\n")
corr.to_csv("correlation_matrix.csv")
print("All answers verified. Results saved to notebook_results.json.")
answers

All answers verified. Results saved to notebook_results.json.


{'Q1': 'technology',
 'Q2': 'interaction_count / lead_score',
 'Q3': 'lead_source',
 'Q4': 0.65,
 'Q5': 'number_of_courses_viewed',
 'Q6': 0.001}